In [1]:
# Prepare the Data for Machine Learning Algorithms
import pandas as pd
import numpy as np
from utils import load_housing_data
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from preprocessingUtils import getPreprocessing

preprocessing = getPreprocessing()
housing = load_housing_data()
housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0.0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5],
)


strat_train_set, strat_test_set = train_test_split(
    housing, test_size=0.2, stratify=housing["income_cat"], random_state=42
)

for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

# Revert to a clean training set
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()



In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression

# Создаём pipeline:
# 1) preprocessing — превращает сырые данные (housing) в 24 числовые фичи
# 2) LinearRegression — обучает модель на этих фичах
lin_reg = make_pipeline(preprocessing, LinearRegression())


# Обучение модели:
# housing — входные данные (признаки районов)
# housing_labels — реальные цены (что нужно предсказать)
# Модель учится находить зависимость: признаки → цена
lin_reg.fit(housing, housing_labels)


# Делаем предсказания на тех же данных (train set)
# Для каждого района считаем его предполагаемую цену
housing_predictions = lin_reg.predict(housing)


# Выводим первые 5 предсказаний
# round(-2) — округление до сотен (чтобы проще читать)
print(
    "predict -\n",
    housing_predictions[:5].round(-2)
)


# Выводим реальные значения (цены) для этих же 5 районов
print(
    "real - \n",
    housing_labels.iloc[:5].values
)

predict -
 [246000. 372700. 135700.  91400. 330900.]
real - 
 [458300. 483800. 101700.  96100. 361800.]


In [5]:
# Measure RMSE 
from sklearn.metrics import root_mean_squared_error
lin_rmse = root_mean_squared_error(housing_labels, housing_predictions)
lin_rmse

68972.88910758478

In [ ]:
# Try DecisionTreeRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import make_pipeline
from sklearn.metrics import root_mean_squared_error

tree_reg = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))
tree_reg.fit(housing, housing_labels)
housing_predictions = tree_reg.predict(housing)
tree_rmse = root_mean_squared_error(housing_labels, housing_predictions)
print(tree_rmse)

0.0


In [ ]:
# Cross validation (считаем ошибку на сколько хорошо)
# k-fold кросс-валидация даёт более надёжную оценку модели:
# train делится на k частей, модель обучается k раз и проверяется на каждой части
from sklearn.model_selection import cross_val_score
tree_rmses = -cross_val_score(tree_reg, housing, housing_labels,
scoring="neg_root_mean_squared_error", cv=10)

pd.Series(tree_rmses).describe()

count       10.000000
mean     67013.360949
std       1460.198570
min      64289.376198
25%      66776.146282
50%      67086.216281
75%      68140.275029
max      68659.294290
dtype: float64

In [ ]:
# Try a RandomForestRegressor
# Random Forest значительно лучше: RMSE ~47k vs ~68k у линейной и дерева
# → более точные и стабильные предсказания
from sklearn.ensemble import RandomForestRegressor

forest_reg = make_pipeline(preprocessing, RandomForestRegressor(random_state=42))
forest_rmses = -cross_val_score(
    forest_reg, housing, housing_labels, scoring="neg_root_mean_squared_error", cv=10
)
pd.Series(forest_rmses).describe()

count       10.000000
mean     47124.604437
std       1069.311372
min      45292.329302
25%      46712.106520
50%      47172.209883
75%      47561.377695
max      49354.705514
dtype: float64